# CS781 Phase 3 - Rhetorical Roles - notebook `B`

**Full design:** sentences (Phase-1 tokenization, <=128 tokens) packed into 512-token windows -> Phase-1 backbone ->
mean-pooled sentence vectors + relative position (16 sinusoids) + CL/IT embedding -> BiLSTM over a range of ~70-100
consecutive sentences (3,072 tokens) -> 13 labels; cross-entropy with sqrt class weights (cap 10).
**Base technique:** LoRA r16 (lr 5e-4). Experiments 15-25 (design ablations, seeds) each change one thing from the base.
Training on CL_train + IT_train only; the epoch (max 10, patience 4) is chosen on CL_dev + IT_dev with the official metric
(macro-F1 over all 13 labels). Dev and test are never trained on.

| ID | GPU | run | technique | change vs full design | question |
|---|---|---|---|---|---|
| B01 | GPU0 | `B01_lora` | LoRA r16 | - | base: the only method that trained stably on all five backbones in the first runs |
| B03 | GPU0 | `B03_lorma_lr2e4` | LoRMA r16 | lr 2e-4 | LoRMA collapsed on A and B at lr 5e-4 in the first runs |
| B05 | GPU0 | `B05_rslora64` | rsLoRA r64 | - | rank-stabilised scaling at a large rank |
| B07 | GPU0 | `B07_qlora` | QLoRA (NF4 + LoRA r16) | - | 4-bit frozen backbone: cost of quantisation |
| B09 | GPU0 | `B09_prompt` | Prompt tuning (20) | - | input-only adaptation, frozen backbone |
| B11 | GPU0 | `B11_adapter` | Houlsby adapter | - | bottleneck adapters |
| B15 | GPU0 | `B15_lora_abl_sentonly` | LoRA r16 | sentence alone, MLP (range 1024) | no context at all |
| B16 | GPU0 | `B16_lora_abl_windowonly` | LoRA r16 | packed windows, MLP head | token-level context only |
| B19 | GPU0 | `B19_lora_abl_noweights` | LoRA r16 | no class weights | class imbalance handling |
| B21 | GPU0 | `B21_lora_abl_ctx6144` | LoRA r16 | range 6144 tokens | longer document context |
| B02 | GPU1 | `B02_lorma` | LoRMA r16 | - | compulsory technique: multiplicative low-rank update |
| B04 | GPU1 | `B04_lorma32` | LoRMA r32 | rank 32 | rank sensitivity of LoRMA |
| B06 | GPU1 | `B06_adalora` | AdaLoRA r12->8 | - | adaptive rank budget across matrices |
| B08 | GPU1 | `B08_ia3` | IA3 | - | ~0.05M-parameter rescaling vs low-rank updates |
| B10 | GPU1 | `B10_prefix` | Prefix tuning (20) | - | learned prefix states at every layer vs input only |
| B12 | GPU1 | `B12_bitfit` | BitFit (norm gains) | - | sparse tuning: only the normalisation gains |
| B13 | GPU1 | `B13_partial` | Partial FT (top 4) | - | layer-wise tuning: top 4 blocks only |
| B14 | GPU1 | `B14_full` | Full FT | - | reference: every weight trains |
| B17 | GPU1 | `B17_lora_abl_singlebilstm` | LoRA r16 | sentence alone + BiLSTM (range 1536) | sentence-level context only |
| B18 | GPU1 | `B18_lora_abl_nopos` | LoRA r16 | no position features | value of the position prior |
| B20 | GPU1 | `B20_lora_abl_lastpool` | LoRA r16 | last-token pooling | sentence representation |

**Inputs:** dataset `cs781-p3-code` (latest version), `cs781-legal-data`, Phase-1 checkpoint of model B.
Internet ON, secret `HF_TOKEN`, GPU **T4 x2**. Save Version -> Save & Run All.
Expected: ~2.7 h after a 3-5 min smoke test. A run that fails the smoke test is dropped (its error is in the
report); the others continue.
**Outputs:** `submissions/<run>.zip` (ready to submit), `models/<run>.pt` (trained parameters + config + best epoch +
dev score), `bundle_B.zip` (dev/test probabilities + reports, for the ensemble), `REPORT_B.txt`.

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading, zipfile
W = "/kaggle/working"
WORK, P3DATA, RUNS, MODELS = f"{W}/p3code", f"{W}/p3data", f"{W}/runs", f"{W}/models"
CACHE = f"{W}/rr_cache.pkl"
for d in (RUNS, MODELS, f"{W}/logs", f"{W}/submissions"):
    os.makedirs(d, exist_ok=True)

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("rr_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

# ---- Phase-3 data from Hugging Face
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(P3DATA, exist_ok=True)
for f in list_repo_files(REPO, repo_type="dataset", token=tok_):
    if f.startswith("Phase_3") and f.endswith(".json"):
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), os.path.join(P3DATA, os.path.basename(f)))
print("p3 data:", sorted(os.listdir(P3DATA)))

# dev truth files in the official evaluator's format (used to cross-check our metric)
import rr_data
for dom in ("CL", "IT"):
    json.dump({x["id"]: x["labels"] for x in rr_data.load_split(f"{P3DATA}/{dom}_dev.json")}, open(f"{W}/dev_truth_{dom}.json", "w"))

# ---- Phase-1 checkpoints (final.pt per model letter; continual-pretrained *_cpt preferred)
CKPT = {}
for p in sorted(glob.glob("/kaggle/input/**/final.pt", recursive=True)):
    m = re.search(r"run_([a-e])_[^/]*", p)
    if not m or "smoke" in p:
        continue
    k = m.group(1)
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

# ---- tokenize every split once (shared by all jobs) + CPU tests
t0 = time.time()
cache = rr_data.build_cache(P3DATA, TOK_ROOT, CACHE)
for k, v in cache["docs"].items():
    print(k, len(v), "docs", sum(len(x["sents"]) for x in v), "sentences")
print(f"cache built in {time.time() - t0:.0f}s")
r = subprocess.run("python tests_rr.py", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"

code: ['adapt.py', 'attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'evaluate_phase3.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'models.py', 'ner_data.py', 'ner_model.py', 'p1_loader.py', 'peft.py', 'rr_data.py', 'rr_ensemble.py', 'rr_eval.py', 'rr_model.py', 'rr_train.py', 'tests_rr.py']


CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

p3 data: ['CL_dev.json', 'CL_test.json', 'CL_train.json', 'IT_dev.json', 'IT_test.json', 'IT_train.json', 'label_vocab.json']
  model b: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/final.pt
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

('CL', 'train') 40 docs 10593 sentences
('CL', 'dev') 5 docs 1396 sentences
('CL', 'test') 5 docs 1339 sentences
('IT', 'train') 40 docs 6252 sentences
('IT', 'dev') 5 docs 746 sentences
('IT', 'test') 5 docs 858 sentences
cache built in 3s
ok test_segments
ok test_pack_and_arrays
ok test_metric
ok test_files_and_validation
 


In [2]:
def cmd(model, out, extra, model_dir):
    return (f"python rr_train.py --ckpt {CKPT[model]} --tok_root {TOK_ROOT} --data_dir {P3DATA} --cache {CACHE} "
            f"--out {out} --model_dir {model_dir} {extra}")

def run(tag, c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    with open(f"{W}/logs/{tag}.txt", "w") as f:
        p = subprocess.run(c, shell=True, cwd=WORK, env=env, stdout=f, stderr=subprocess.STDOUT)
    tail = open(f"{W}/logs/{tag}.txt").read().strip().splitlines()[-2:]
    print(f"[GPU{gpu}] {tag}: exit {p.returncode} | " + " / ".join(tail)[-300:], flush=True)
    return p.returncode

def run_queues(queues, smoke=False):
    """queues: one list of experiments per GPU, run in parallel. smoke: 1 tiny epoch per experiment."""
    status = {}
    def worker(gpu, jobs):
        for e in jobs:
            t = ("smoke_" if smoke else "") + e["tag"]
            extra = e["args"] + (" --max_docs 1 --max_sents 40 --epochs 1" if smoke else "")
            if smoke and e is not jobs[0]:
                extra += " --no_test"
            status[t] = run(t, cmd(e["model"], f"{RUNS}/{t}", extra, f"{W}/smoke_models" if smoke else MODELS), gpu)
    th = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues) if q]
    t0 = time.time()
    for t in th: t.start()
    for t in th: t.join()
    print(f"{'smoke' if smoke else 'runs'} done in {(time.time() - t0) / 60:.1f} min:", status)
    return status

def smoke(queues):
    """Runs 1 tiny epoch of every experiment. A failing experiment is dropped (its error is printed and
    reported) instead of stopping the notebook; the notebook stops only if every experiment fails."""
    st = run_queues(queues, smoke=True)
    bad = [k for k, v in st.items() if v != 0]
    for k in bad:
        print(f"===== SMOKE FAILED, dropped: {k}\n" + open(f"{W}/logs/{k}.txt").read()[-3000:])
    assert len(bad) < len(st), "every experiment failed the smoke test"
    for k in st:
        shutil.rmtree(f"{RUNS}/{k}", ignore_errors=True)
    shutil.rmtree(f"{W}/smoke_models", ignore_errors=True)
    print("SMOKE OK" if not bad else f"SMOKE OK except {len(bad)} dropped: {bad}")
    return [[e for e in q if "smoke_" + e["tag"] not in bad] for q in queues]

## Experiments (one queue per GPU)

In [3]:
NAME = 'B'
QUEUES = [[{'id': 'B01', 'tag': 'B01_lora', 'model': 'b', 'tech': 'LoRA r16', 'change': '-', 'args': '--method lora --rank 16 --alpha 32'},
  {'id': 'B03',
   'tag': 'B03_lorma_lr2e4',
   'model': 'b',
   'tech': 'LoRMA r16',
   'change': 'lr 2e-4',
   'args': '--method lorma --rank 16 --alpha 32 --lr 2e-4'},
  {'id': 'B05', 'tag': 'B05_rslora64', 'model': 'b', 'tech': 'rsLoRA r64', 'change': '-', 'args': '--method rslora --rank 64 --alpha 16'},
  {'id': 'B07', 'tag': 'B07_qlora', 'model': 'b', 'tech': 'QLoRA (NF4 + LoRA r16)', 'change': '-', 'args': '--method qlora --rank 16 --alpha 32'},
  {'id': 'B09', 'tag': 'B09_prompt', 'model': 'b', 'tech': 'Prompt tuning (20)', 'change': '-', 'args': '--method prompt --n_prompt 20'},
  {'id': 'B11', 'tag': 'B11_adapter', 'model': 'b', 'tech': 'Houlsby adapter', 'change': '-', 'args': '--method adapter --bottleneck 64'},
  {'id': 'B15',
   'tag': 'B15_lora_abl_sentonly',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': 'sentence alone, MLP (range 1024)',
   'args': '--method lora --rank 16 --alpha 32 --mode single --head mlp --seg_tokens 1024'},
  {'id': 'B16',
   'tag': 'B16_lora_abl_windowonly',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': 'packed windows, MLP head',
   'args': '--method lora --rank 16 --alpha 32 --head mlp'},
  {'id': 'B19',
   'tag': 'B19_lora_abl_noweights',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': 'no class weights',
   'args': '--method lora --rank 16 --alpha 32 --class_weight none'},
  {'id': 'B21',
   'tag': 'B21_lora_abl_ctx6144',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': 'range 6144 tokens',
   'args': '--method lora --rank 16 --alpha 32 --seg_tokens 6144'}],
 [{'id': 'B02', 'tag': 'B02_lorma', 'model': 'b', 'tech': 'LoRMA r16', 'change': '-', 'args': '--method lorma --rank 16 --alpha 32'},
  {'id': 'B04', 'tag': 'B04_lorma32', 'model': 'b', 'tech': 'LoRMA r32', 'change': 'rank 32', 'args': '--method lorma --rank 32 --alpha 64'},
  {'id': 'B06',
   'tag': 'B06_adalora',
   'model': 'b',
   'tech': 'AdaLoRA r12->8',
   'change': '-',
   'args': '--method adalora --rank 12 --alpha 24 --target_rank 8'},
  {'id': 'B08', 'tag': 'B08_ia3', 'model': 'b', 'tech': 'IA3', 'change': '-', 'args': '--method ia3'},
  {'id': 'B10', 'tag': 'B10_prefix', 'model': 'b', 'tech': 'Prefix tuning (20)', 'change': '-', 'args': '--method prefix --n_prompt 20'},
  {'id': 'B12', 'tag': 'B12_bitfit', 'model': 'b', 'tech': 'BitFit (norm gains)', 'change': '-', 'args': '--method norm'},
  {'id': 'B13', 'tag': 'B13_partial', 'model': 'b', 'tech': 'Partial FT (top 4)', 'change': '-', 'args': '--method partial --top_k 4'},
  {'id': 'B14', 'tag': 'B14_full', 'model': 'b', 'tech': 'Full FT', 'change': '-', 'args': '--method full'},
  {'id': 'B17',
   'tag': 'B17_lora_abl_singlebilstm',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': 'sentence alone + BiLSTM (range 1536)',
   'args': '--method lora --rank 16 --alpha 32 --mode single --seg_tokens 1536'},
  {'id': 'B18',
   'tag': 'B18_lora_abl_nopos',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': 'no position features',
   'args': '--method lora --rank 16 --alpha 32 --no_pos'},
  {'id': 'B20',
   'tag': 'B20_lora_abl_lastpool',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': 'last-token pooling',
   'args': '--method lora --rank 16 --alpha 32 --pool last'}]]
ALL_QUEUES = [list(q) for q in QUEUES]                    # the full plan (the report lists dropped runs too)

## 1. Smoke test

In [4]:
QUEUES = smoke(QUEUES)

[GPU1] smoke_B02_lorma: exit 0 | ion.zip  {'Fact': 2150, 'Issue': 0, 'ArgumentPetitioner': 0, 'ArgumentRespondent': 0, 'Statute': 0, 'Dissent': 0, 'PrecedentReliedUpon': 0, 'PrecedentNotReliedUpon': 0, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 47, 'RatioOfTheDecision': 0, 'RulingByPresentCourt': 0, 'None': 0} / [06:14:30] DONE
[GPU0] smoke_B01_lora: exit 0 | sion.zip  {'Fact': 2197, 'Issue': 0, 'ArgumentPetitioner': 0, 'ArgumentRespondent': 0, 'Statute': 0, 'Dissent': 0, 'PrecedentReliedUpon': 0, 'PrecedentNotReliedUpon': 0, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 0, 'RatioOfTheDecision': 0, 'RulingByPresentCourt': 0, 'None': 0} / [06:14:30] DONE
[GPU1] smoke_B04_lorma32: exit 0 | confusions ArgumentPetitioner->Fact:16 RatioOfTheDecision->Fact:16 RulingByLowerCourt->Fact:4 None->Fact:3 Issue->Fact:3 Statute->Fact:1 / [06:14:41] DONE
[GPU0] smoke_B03_lorma_lr2e4: exit 0 | confusions ArgumentPetitioner->Fact:16 RatioOfTheDecision->Fact:15 RulingByLowerCourt->Fact:4 None->Fact

## 2. Runs

In [5]:
status = run_queues(QUEUES)

[GPU1] B02_lorma: exit 0 | act': 267, 'Issue': 32, 'ArgumentPetitioner': 136, 'ArgumentRespondent': 332, 'Statute': 138, 'Dissent': 0, 'PrecedentReliedUpon': 193, 'PrecedentNotReliedUpon': 0, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 8, 'RatioOfTheDecision': 1019, 'RulingByPresentCourt': 54, 'None': 18} / [06:23:05] DONE
[GPU0] B01_lora: exit 0 | ct': 376, 'Issue': 17, 'ArgumentPetitioner': 277, 'ArgumentRespondent': 36, 'Statute': 162, 'Dissent': 6, 'PrecedentReliedUpon': 389, 'PrecedentNotReliedUpon': 35, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 131, 'RatioOfTheDecision': 707, 'RulingByPresentCourt': 38, 'None': 23} / [06:28:18] DONE
[GPU1] B04_lorma32: exit 0 | ct': 442, 'Issue': 19, 'ArgumentPetitioner': 241, 'ArgumentRespondent': 68, 'Statute': 156, 'Dissent': 0, 'PrecedentReliedUpon': 323, 'PrecedentNotReliedUpon': 22, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 103, 'RatioOfTheDecision': 766, 'RulingByPresentCourt': 32, 'None': 25} / [06:37:17] DONE
[GPU0] B03_lorm

## 3. Report

In [6]:
SHORT = {"ArgumentPetitioner": "ArgPet", "ArgumentRespondent": "ArgResp", "PrecedentReliedUpon": "PrecRel",
         "PrecedentNotReliedUpon": "PrecNRel", "PrecedentOverruled": "PrecOvr", "RulingByLowerCourt": "RulLower",
         "RatioOfTheDecision": "Ratio", "RulingByPresentCourt": "RulPres"}

def official(tag):
    """Official evaluator on this run's dev predictions (cross-check of our metric)."""
    r = subprocess.run(f"python evaluate_phase3.py --truth-cl {W}/dev_truth_CL.json --truth-it {W}/dev_truth_IT.json "
                       f"--prediction {RUNS}/{tag}/dev_pred.json --labels {P3DATA}/label_vocab.json",
                       shell=True, capture_output=True, text=True, cwd=WORK)
    m = re.search(r"Macro-F1:\s+([0-9.]+)", r.stdout)
    return float(m.group(1)) if m else float("nan")

def log_tail(name, n=2):
    p = f"{W}/logs/{name}.txt"
    return " / ".join(open(p).read().strip().splitlines()[-n:])[-250:] if os.path.exists(p) else ""

def summarize(name, queues):
    exps = [e for q in queues for e in q]
    L = [f"=== REPORT {name} ===",
         "ID  | run                          | technique          | change                     | dev F1 | official | CL     | IT     | P      | R      | ep | trainable | min  | tok/s  | mem"]
    rows = []
    for e in exps:
        p = f"{RUNS}/{e['tag']}/summary.json"
        if not os.path.exists(p):
            why = "SMOKE FAILED (dropped)" if not os.path.exists(f"{W}/logs/{e['tag']}.txt") else "FAILED"
            L.append(f"{e['id']} | {e['tag']:28s} | {why}: {log_tail(e['tag']) or log_tail('smoke_' + e['tag'])}")
            continue
        s = json.load(open(p)); dv = s["dev"]
        off = official(e["tag"])
        rows.append((e, s))
        L.append(f"{e['id']} | {e['tag']:28s} | {e['tech'][:18]:18s} | {e['change'][:26]:26s} | {dv['macro_f1']:.4f} | {off:.4f}   | "
                 f"{dv['macro_f1_CL']:.4f} | {dv['macro_f1_IT']:.4f} | {dv['macro_p']:.4f} | {dv['macro_r']:.4f} | {s['best_epoch']:2d} | "
                 f"{s['params_trainable'] / 1e6:8.2f}M | {s['train_minutes']:4.0f} | {s.get('tok_s') or 0:6,.0f} | {s.get('peak_mem_gb') or 0:.1f}G")
        shutil.copy(f"{RUNS}/{e['tag']}/submission.zip", f"{W}/submissions/{e['tag']}.zip")
    if rows:
        labs = rows[0][1]["labels"]
        L.append("\nper-label dev F1 (official: all 13 labels count; Dissent and PrecedentOverruled have no dev sentences -> 0)")
        L.append(f"{'ID':4s} " + " ".join(f"{SHORT.get(l, l)[:8]:>8s}" for l in labs))
        for e, s in rows:
            pl = s["dev"]["per_label"]
            L.append(f"{e['id']:4s} " + " ".join(f"{pl[l]['f1']:8.2f}" for l in labs))
        L.append("\ntop confusions (gold->pred:count)")
        for e, s in rows:
            L.append(f"{e['id']}: " + " ".join(s["dev"]["top_confusions"][:6]))
    bundle = f"{W}/bundle_{name}.zip"                     # everything except model weights (those stay in models/)
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for e in exps:
            d = f"{RUNS}/{e['tag']}"
            if os.path.isdir(d):
                for f in os.listdir(d):
                    z.write(f"{d}/{f}", arcname=f"{e['tag']}/{f}")
    L.append(f"\nbundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")
    L.append("models: " + ", ".join(f"{f} ({os.path.getsize(f'{MODELS}/{f}') / 2**20:.0f} MB)" for f in sorted(os.listdir(MODELS))))
    L.append("submission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
    rep = "\n".join(L)
    open(f"{W}/REPORT_{name}.txt", "w").write(rep)
    print(rep + "\n=== END REPORT ===")

summarize(NAME, ALL_QUEUES)

=== REPORT B ===
ID  | run                          | technique          | change                     | dev F1 | official | CL     | IT     | P      | R      | ep | trainable | min  | tok/s  | mem
B01 | B01_lora                     | LoRA r16           | -                          | 0.5110 | 0.5110   | 0.4403 | 0.5207 | 0.5205 | 0.5191 |  5 |     4.68M |   12 |  8,614 | 3.4G
B03 | B03_lorma_lr2e4              | LoRMA r16          | lr 2e-4                    | 0.5196 | 0.5196   | 0.4101 | 0.5278 | 0.5493 | 0.5089 |  6 |     5.22M |   13 |  8,599 | 3.8G
B05 | B05_rslora64                 | rsLoRA r64         | -                          | 0.4886 | 0.4886   | 0.4178 | 0.4938 | 0.4971 | 0.4921 |  5 |    12.20M |   12 |  8,405 | 3.6G
B07 | B07_qlora                    | QLoRA (NF4 + LoRA  | -                          | 0.5075 | 0.5075   | 0.3963 | 0.4996 | 0.5396 | 0.5005 |  6 |     4.68M |   15 |  7,845 | 3.0G
B09 | B09_prompt                   | Prompt tuning (20) | -                    